# Лабораторная работа: Кластеризация и оценка качества в обучении без учителя

*ML-4.1 Способен уверенно применять инструменты очистки данных и предварительной подготовки данных для анализа данных методами МО без учителя (методы кластеризации, понижения размерности и визуализации) для анализа данных и выявления скрытых закономерностей с учетом сложности задачи и масштаба данных*

*ML-4.3 Способен комбинировать методы обучения без учителя для комплексного анализа данных, уверенно применять метрики качества обучения без учителя (коэффициент силуэта, скорректированный индекс Рэнда и др.) и достоверно интерпретировать полученные результаты в контексте предметной области*

Сделайте копию этого файла (Файл - Сохранить копию на диске), переименуйте её, добавив в название вашу фамилию. Например, Иванова_Кластеризация.ipynb

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    adjusted_rand_score, adjusted_mutual_info_score,
    davies_bouldin_score, calinski_harabasz_score
)
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Часть 1. Генерация и визуализация данных

Для демонстрации различных методов кластеризации используем несколько синтетических датасетов с разной структурой.

In [ ]:
# Генерация трех датасетов

# 1. Хорошо разделимые сферические кластеры
X_blobs, y_blobs = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)

# 2. Нелинейно разделимые данные (два полумесяца)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# 3. Данные с аномалиями (blobs + шум)
X_blobs_noise, _ = make_blobs(n_samples=400, centers=3, cluster_std=0.8, random_state=42)
X_noise = np.random.uniform(-8, 8, (50, 2))
X_anomaly = np.vstack([X_blobs_noise, X_noise])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], alpha=0.7)
axes[0].set_title('Хорошо разделимые кластеры')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], alpha=0.7)
axes[1].set_title('Нелинейная структура (два полумесяца)')
axes[2].scatter(X_anomaly[:, 0], X_anomaly[:, 1], alpha=0.7, color='red')
axes[2].set_title('Кластеры с аномалиями')
for ax in axes:
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print(f"X_blobs: {X_blobs.shape}, X_moons: {X_moons.shape}, X_anomaly: {X_anomaly.shape}")

**Практика.**

1. Визуализируйте данные с помощью цветовой кодировки.
2. Как вы думаете, какие методы кластеризации хорошо справятся с каждым типом данных?
3. Почему метод K-means может не справиться с данными в виде полумесяцев?

In [ ]:
# Разместите здесь ваш код

# 1. Визуализация данных с цветами

**Ваш ответ на вопросы:**

# Часть 2. Метод K-means

## 2.1. Базовое применение K-means

K-means — один из самых популярных алгоритмов кластеризации. Он разбивает данные на K кластеров, минимизируя внутрикластерную сумму квадратов (WCSS).

In [ ]:
# Применяем K-means к данным blobs
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_kmeans = kmeans.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Исходные данные
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', alpha=0.7)
axes[0].set_title('Истинная разметка')

# Результат кластеризации
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_kmeans, cmap='viridis', alpha=0.7)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                c='red', marker='X', s=200, label='Центры кластеров')
axes[1].set_title('K-means (K=4)')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"WCSS (inertia): {kmeans.inertia_:.2f}")

**Практика.**

1. Примените K-means к данным X_moons с K=2. Визуализируйте результат.
2. Объясните, почему K-means не справляется с данными такой формы.
3. Какой метод кластеризации лучше подходит для таких данных?

In [ ]:
# Разместите здесь ваш код

# K-means на X_moons

**Ваш ответ на вопросы:**

## 2.2. E-means (Gaussian Mixture Models)

Gaussian Mixture Models (GMM) — вероятностная версия K-means. Вместо жесткого назначения точек кластерам, GMM оценивает вероятность принадлежности точки каждому кластеру. Это позволяет моделировать кластеры эллиптической формы.

In [ ]:
# Применяем GMM к данным
gmm = GaussianMixture(n_components=4, random_state=42)
y_gmm = gmm.fit_predict(X_blobs)

# Вероятности принадлежности кластерам
probs = gmm.predict_proba(X_blobs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', alpha=0.7)
axes[0].set_title('Истинная разметка')

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_gmm, cmap='viridis', alpha=0.7)
axes[1].set_title('GMM (K=4)')

# Визуализация неопределенности
uncertainty = -np.max(probs, axis=1)
scatter = axes[2].scatter(X_blobs[:, 0], X_blobs[:, 1], c=uncertainty, cmap='hot', alpha=0.7)
axes[2].set_title('Неопределенность классификации (чем выше, тем темнее)')
plt.colorbar(scatter, ax=axes[2])

plt.tight_layout()
plt.show()

**Практика.**

1. Примените GMM к данным X_moons с K=2. Сравните с K-means.
2. Визуализируйте вероятности принадлежности кластерам.
3. В чем ключевое отличие GMM от K-means?

In [ ]:
# Разместите здесь ваш код

# GMM на X_moons

**Ваш ответ на вопросы:**

# Часть 3. Иерархическая кластеризация и дендрограмма

Иерархическая кластеризация строит дерево (дендрограмму) кластеров. Это позволяет визуально оценить структуру данных и выбрать число кластеров.

**Методы связи (linkage):**
- `single` — минимальное расстояние между кластерами
- `complete` — максимальное расстояние между кластерами
- `average` — среднее расстояние между кластерами
- `ward` — минимизация внутрикластерной дисперсии

In [ ]:
# Построение дендрограммы для данных blobs
linked = linkage(X_blobs, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linked, truncate_mode='lastp', p=20, leaf_rotation=90, leaf_font_size=10)
plt.title('Дендрограмма (Ward linkage)')
plt.xlabel('Индекс объекта')
plt.ylabel('Расстояние')
plt.axhline(y=25, color='r', linestyle='--', label='Горизонтальное сечение (K=4)')
plt.legend()
plt.tight_layout()
plt.show()

**Практика.**

1. Постройте дендрограмму для X_moons с разными методами связи (ward, complete, single).
2. Сравните дендрограммы. Какой метод лучше отражает структуру данных?
3. Какой метод связи (linkage) лучше всего подходит для данных с шумом?

In [ ]:
# Разместите здесь ваш код

# Дендрограммы для X_moons

**Ваш ответ на вопросы:**

# Часть 4. Оценка качества кластеризации

## 4.1. Основные метрики

В обучении без учителя метрики качества делятся на два типа:

1. **Внешние метрики** — требуют истинной разметки:
   - Adjusted Rand Index (ARI)
   - Adjusted Mutual Information (AMI)

2. **Внутренние метрики** — не требуют разметки:
   - Silhouette Score
   - Davies-Bouldin Index
   - Calinski-Harabasz Index
   - WCSS (Inertia)

In [ ]:
# Функция для оценки кластеризации
def evaluate_clustering(X, labels_true, labels_pred, name=''):
    """Оценка качества кластеризации"""
    
    # Внешние метрики (сравнение с истиной)
    ari = adjusted_rand_score(labels_true, labels_pred)
    ami = adjusted_mutual_info_score(labels_true, labels_pred)
    
    # Внутренние метрики
    silhouette = silhouette_score(X, labels_pred) if len(np.unique(labels_pred)) > 1 else np.nan
    davies_bouldin = davies_bouldin_score(X, labels_pred) if len(np.unique(labels_pred)) > 1 else np.nan
    calinski = calinski_harabasz_score(X, labels_pred) if len(np.unique(labels_pred)) > 1 else np.nan
    
    print(f"\n{'='*50}")
    print(f"Оценка кластеризации: {name}")
    print(f"{'='*50}")
    print(f"Внешние метрики:")
    print(f"  Adjusted Rand Index (ARI):        {ari:.4f}")
    print(f"  Adjusted Mutual Info (AMI):       {ami:.4f}")
    print(f"Внутренние метрики:")
    print(f"  Silhouette Score:                 {silhouette:.4f}")
    print(f"  Davies-Bouldin Index:             {davies_bouldin:.4f}")
    print(f"  Calinski-Harabasz Index:          {calinski:.2f}")
    
    return {'ARI': ari, 'AMI': ami, 'Silhouette': silhouette, 
            'Davies-Bouldin': davies_bouldin, 'Calinski-Harabasz': calinski}

In [ ]:
# Применяем разные методы к X_blobs

# K-means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_blobs)

# GMM
gmm = GaussianMixture(n_components=4, random_state=42)
labels_gmm = gmm.fit_predict(X_blobs)

# Агломеративная кластеризация
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_agg = agg.fit_predict(X_blobs)

# Оценка
scores_kmeans = evaluate_clustering(X_blobs, y_blobs, labels_kmeans, 'K-means')
scores_gmm = evaluate_clustering(X_blobs, y_blobs, labels_gmm, 'GMM')
scores_agg = evaluate_clustering(X_blobs, y_blobs, labels_agg, 'Agglomerative')

**Практика.**

1. Сравните метрики качества для K-means, GMM и агломеративной кластеризации.
2. Какой метод показал лучшие результаты? Почему?
3. Какая внутренняя метрика лучше всего отражает качество кластеризации на этих данных?

In [ ]:
# Разместите здесь ваш код

# Визуализация сравнения метрик

**Ваш ответ на вопросы:**

## 4.2. Силуэтный анализ

Силуэтный анализ показывает, насколько хорошо каждый объект соответствует своему кластеру. Значения близкие к 1 указывают на хорошую кластеризацию, близкие к 0 — на пересечение кластеров, отрицательные — на неправильную кластеризацию.

In [ ]:
# Силуэтный анализ для K-means
silhouette_vals = silhouette_samples(X_blobs, labels_kmeans)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Силуэтная диаграмма
y_lower = 10
for i in range(4):
    ith_cluster_silhouette_vals = silhouette_vals[labels_kmeans == i]
    ith_cluster_silhouette_vals.sort()
    size_cluster_i = ith_cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    color = plt.cm.viridis(float(i) / 4)
    axes[0].fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_vals,
                         facecolor=color, edgecolor=color, alpha=0.7)
    axes[0].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    y_lower = y_upper + 10

axes[0].axvline(x=silhouette_score(X_blobs, labels_kmeans), color='red', linestyle='--', label='Средний силуэт')
axes[0].set_title('Силуэтная диаграмма для K-means')
axes[0].set_xlabel('Коэффициент силуэта')
axes[0].set_ylabel('Кластер')
axes[0].legend()

# Визуализация с цветами по силуэту
scatter = axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=silhouette_vals, cmap='plasma', alpha=0.7)
axes[1].set_title('Значения силуэта для каждого объекта')
plt.colorbar(scatter, ax=axes[1])

plt.tight_layout()
plt.show()

print(f"Средний коэффициент силуэта: {silhouette_score(X_blobs, labels_kmeans):.4f}")
print(f"Минимальный коэффициент силуэта: {silhouette_vals.min():.4f}")
print(f"Максимальный коэффициент силуэта: {silhouette_vals.max():.4f}")

**Практика.**

1. Постройте силуэтную диаграмму для K-means на X_moons с K=2.
2. Сравните с результатами на X_blobs. Что изменилось?
3. Как интерпретировать отрицательные значения коэффициента силуэта?

In [ ]:
# Разместите здесь ваш код

# Силуэтный анализ для X_moons

**Ваш ответ на вопросы:**

# Часть 5. Выбор числа кластеров

## 5.1. Метод локтя (Elbow Method)

Метод локтя основан на анализе WCSS (within-cluster sum of squares). Оптимальное число кластеров находится в точке "излома" графика.

In [ ]:
# Метод локтя для X_blobs
wcss = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_blobs)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, wcss, 'bo-', linewidth=2)
plt.axvline(x=4, color='red', linestyle='--', label='Оптимальное K=4')
plt.xlabel('Число кластеров K')
plt.ylabel('WCSS (Inertia)')
plt.title('Метод локтя для определения оптимального K')
plt.legend()
plt.grid(True)
plt.show()

## 5.2. Силуэтный анализ для выбора K

Средний коэффициент силуэта для разных K позволяет выбрать оптимальное число кластеров.

In [ ]:
# Силуэтный анализ для разных K
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_blobs)
    silhouette_scores.append(silhouette_score(X_blobs, labels))

plt.figure(figsize=(8, 5))
plt.plot(K_range, silhouette_scores, 'ro-', linewidth=2)
plt.axvline(x=4, color='blue', linestyle='--', label='Оптимальное K=4')
plt.xlabel('Число кластеров K')
plt.ylabel('Средний коэффициент силуэта')
plt.title('Силуэтный анализ для выбора K')
plt.legend()
plt.grid(True)
plt.show()

**Практика.**

1. Примените метод локтя и силуэтный анализ к X_moons.
2. Какое K рекомендуется каждым методом?
3. Какой метод более надежен для выбора числа кластеров? Почему?

In [ ]:
# Разместите здесь ваш код

# Методы выбора K для X_moons

**Ваш ответ на вопросы:**

# Часть 6. DBSCAN и детектирование аномалий

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) — алгоритм кластеризации, основанный на плотности. Он позволяет находить кластеры произвольной формы и идентифицировать аномалии.

**Ключевые параметры:**
- `eps` — радиус окрестности
- `min_samples` — минимальное число точек для формирования кластера

In [ ]:
# Применяем DBSCAN к данным
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_anomaly)

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Результат кластеризации
unique_labels = set(labels_dbscan)
colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    mask = labels_dbscan == label
    if label == -1:
        # Аномалии
        axes[0].scatter(X_anomaly[mask, 0], X_anomaly[mask, 1], color='red', alpha=0.7, label='Аномалии', s=50)
    else:
        axes[0].scatter(X_anomaly[mask, 0], X_anomaly[mask, 1], color=color, alpha=0.7, label=f'Кластер {label}')

axes[0].set_title('DBSCAN (eps=0.5, min_samples=5)')
axes[0].legend()

# Гистограмма распределения по кластерам
cluster_counts = pd.Series(labels_dbscan).value_counts().sort_index()
axes[1].bar(cluster_counts.index.astype(str), cluster_counts.values, color=['red' if i == -1 else 'blue' for i in cluster_counts.index])
axes[1].set_title('Распределение объектов по кластерам')
axes[1].set_xlabel('Кластер')
axes[1].set_ylabel('Число объектов')

plt.tight_layout()
plt.show()

n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise = list(labels_dbscan).count(-1)
print(f"Число кластеров: {n_clusters}")
print(f"Число аномалий: {n_noise}")

**Практика.**

1. Измените параметры DBSCAN (eps, min_samples) на X_anomaly.
2. Как влияет изменение eps на число кластеров и аномалий?
3. Как влияет изменение min_samples на число кластеров и аномалий?
4. При каких параметрах DBSCAN лучше всего отделяет аномалии от основных кластеров?

In [ ]:
# Разместите здесь ваш код

# Эксперименты с параметрами DBSCAN

**Ваш ответ на вопросы:**

# Часть 7. Применение кластеризации для разметки данных

Кластеризация может использоваться для автоматической разметки данных, когда у нас нет истинных меток.

## 7.1. Кластеризация для разметки данных

Кластеризация может использоваться для автоматической разметки данных, когда у нас нет истинных меток. Рассмотрим задачу: у нас есть набор изображений рукописных цифр, но мы не знаем, какая цифра на каком изображении. Используем кластеризацию для разметки.

In [ ]:
# Загружаем датасет digits
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# Стандартизация
scaler = StandardScaler()
X_digits_scaled = scaler.fit_transform(X_digits)

# Применяем PCA для визуализации
pca = PCA(n_components=2)
X_digits_pca = pca.fit_transform(X_digits_scaled)

# K-means кластеризация
kmeans_digits = KMeans(n_clusters=10, random_state=42, n_init=10)
labels_digits = kmeans_digits.fit_predict(X_digits_scaled)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Истинные метки
axes[0].scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], c=y_digits, cmap='tab10', alpha=0.7)
axes[0].set_title('Истинные метки (цифры)')

# Кластеры K-means
axes[1].scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], c=labels_digits, cmap='tab10', alpha=0.7)
axes[1].set_title('Кластеры K-means (K=10)')

# Визуализация центров кластеров
centers_pca = pca.transform(kmeans_digits.cluster_centers_)
axes[2].scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], c=labels_digits, cmap='tab10', alpha=0.3)
axes[2].scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', marker='X', s=200, label='Центры кластеров')
axes[2].set_title('Центры кластеров в пространстве PCA')
axes[2].legend()

plt.tight_layout()
plt.show()

# Оценка качества (сравнение с истинными метками)
ari_digits = adjusted_rand_score(y_digits, labels_digits)
ami_digits = adjusted_mutual_info_score(y_digits, labels_digits)
silhouette_digits = silhouette_score(X_digits_scaled, labels_digits)

print(f"Adjusted Rand Index (ARI):     {ari_digits:.4f}")
print(f"Adjusted Mutual Info (AMI):    {ami_digits:.4f}")
print(f"Silhouette Score:              {silhouette_digits:.4f}")

**Практика.**

1. Что показывают метрики ARI и AMI? Почему они не равны 1.0, если K=10 соответствует числу классов?
2. Как вы думаете, почему некоторые кластеры соответствуют нескольким цифрам?
3. Как можно улучшить качество кластеризации для этого датасета?

In [ ]:
# Разместите здесь ваш код

# Эксперименты с улучшением кластеризации digits

**Ваш ответ на вопросы:**

## 7.2. Интерпретация латентных параметров

Латентные параметры — это скрытые характеристики данных, которые алгоритм кластеризации выявляет. Например, в датасете цифр это могут быть различные стили написания, наклон, толщина линий.

Рассмотрим, как интерпретировать центры кластеров в пространстве признаков.

In [ ]:
# Визуализация центров кластеров как изображений цифр
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    # Восстанавливаем центр кластера в исходное пространство
    center_image = kmeans_digits.cluster_centers_[i].reshape(8, 8)
    ax.imshow(center_image, cmap='gray')
    
    # Находим, какая цифра доминирует в этом кластере
    mask = labels_digits == i
    if np.sum(mask) > 0:
        dominant_digit = np.bincount(y_digits[mask]).argmax()
        count = np.bincount(y_digits[mask]).max()
        ax.set_title(f'Кластер {i}\nДоминирует: {dominant_digit}\n({count} объектов)')
    else:
        ax.set_title(f'Кластер {i}\n(пустой)')
    ax.axis('off')

plt.tight_layout()
plt.show()

**Практика.**

1. Визуализируйте центры кластеров для K=15.
2. Какие стили написания цифр можно увидеть в разных кластерах?
3. Как интерпретировать латентные параметры в контексте задачи распознавания цифр?
4. Может ли кластерный анализ помочь в обнаружении новых, неизвестных заранее классов?

In [ ]:
# Разместите здесь ваш код

# Визуализация центров кластеров для K=15

**Ваш ответ на вопросы:**

# Часть 8. Сравнительный анализ методов кластеризации

## 8.1. Сравнение всех методов на разных типах данных

Сравним все изученные методы на трех типах данных и оценим их качество.

In [ ]:
# Функция для сравнения методов
def compare_clustering_methods(X, y_true, name=''):
    """Сравнение методов кластеризации"""
    
    methods = {
        'K-means (K=4)': KMeans(n_clusters=4, random_state=42, n_init=10),
        'GMM (K=4)': GaussianMixture(n_components=4, random_state=42),
        'Agglomerative (ward)': AgglomerativeClustering(n_clusters=4, linkage='ward'),
        'DBSCAN (eps=0.5)': DBSCAN(eps=0.5, min_samples=5)
    }
    
    results = {}
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    
    for idx, (method_name, model) in enumerate(methods.items()):
        try:
            labels = model.fit_predict(X)
            
            # Пропускаем если один кластер или все аномалии
            if len(np.unique(labels)) <= 1:
                results[method_name] = {'ARI': np.nan, 'AMI': np.nan, 
                                       'Silhouette': np.nan, 'Clusters': len(np.unique(labels))}
                axes[idx].text(0.5, 0.5, 'Не удалось найти кластеры', 
                              ha='center', va='center', transform=axes[idx].transAxes)
                axes[idx].set_title(method_name)
                continue
            
            # Метрики
            ari = adjusted_rand_score(y_true, labels)
            ami = adjusted_mutual_info_score(y_true, labels)
            silhouette = silhouette_score(X, labels) if len(np.unique(labels)) > 1 else np.nan
            
            results[method_name] = {'ARI': ari, 'AMI': ami, 
                                   'Silhouette': silhouette, 'Clusters': len(np.unique(labels))}
            
            # Визуализация
            scatter = axes[idx].scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', alpha=0.7)
            axes[idx].set_title(f'{method_name}\nARI={ari:.3f}, Silhouette={silhouette:.3f}')
            
        except Exception as e:
            results[method_name] = {'ARI': np.nan, 'AMI': np.nan, 
                                   'Silhouette': np.nan, 'Clusters': 0}
            axes[idx].text(0.5, 0.5, f'Ошибка: {str(e)[:50]}', 
                          ha='center', va='center', transform=axes[idx].transAxes)
            axes[idx].set_title(method_name)
    
    plt.suptitle(f'Сравнение методов кластеризации на данных: {name}', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Таблица результатов
    df_results = pd.DataFrame(results).T
    print(f"\n{'='*60}")
    print(f"Результаты на данных: {name}")
    print(f"{'='*60}")
    print(df_results.round(4))
    
    return df_results

In [ ]:
# Сравнение на X_blobs
results_blobs = compare_clustering_methods(X_blobs, y_blobs, 'X_blobs')

# Сравнение на X_moons
results_moons = compare_clustering_methods(X_moons, y_moons, 'X_moons')

# Сравнение на X_anomaly
results_anomaly = compare_clustering_methods(X_anomaly, np.zeros(len(X_anomaly)), 'X_anomaly')

**Практика.**

1. Какой метод показал лучшие результаты на каждом типе данных?
2. Почему DBSCAN лучше справляется с X_moons, чем K-means?
3. Почему на X_anomaly DBSCAN показывает хорошие результаты, а другие методы нет?
4. Сделайте вывод: для каких задач какой метод кластеризации предпочтительнее.

In [ ]:
# Разместите здесь ваш код

# Дополнительный анализ результатов

**Ваш ответ на вопросы:**

# Часть 9. Итоговое задание: Комплексный анализ данных

**Практика.**

Выберите один из реальных датасетов (например, load_wine или load_digits) и выполните комплексный анализ:

1. **Предобработка данных:** стандартизация, удаление выбросов (по желанию)
2. **Визуализация:** PCA, t-SNE для понимания структуры данных
3. **Кластеризация:** примените 3-4 различных метода кластеризации
4. **Выбор числа кластеров:** метод локтя, силуэтный анализ
5. **Оценка качества:** вычислите внутренние и внешние (если есть метки) метрики
6. **Интерпретация:** опишите, какие группы (кластеры) вы обнаружили и как их можно интерпретировать в контексте предметной области

**Вопросы для анализа:**
- Совпадают ли кластеры с истинными классами (если есть метки)?
- Какие объекты оказались в "пограничных" зонах между кластерами?
- Можно ли использовать кластеризацию для обнаружения новых, неизвестных ранее закономерностей в данных?
- Какие ограничения есть у каждого из использованных методов?

In [ ]:
from sklearn.datasets import load_wine

# Разместите здесь ваш код

# Загрузка данных
data = load_wine()
X_real = data.data
y_real = data.target

# 1. Предобработка

# 2. Визуализация

# 3. Кластеризация

# 4. Выбор числа кластеров

# 5. Оценка качества

# 6. Интерпретация

**Ваш ответ на вопросы анализа:**

# Итоговые выводы

**Практика.**

Напишите эссе (5-7 предложений), в котором:
1. Сравните изученные методы кластеризации (K-means, GMM, иерархическая, DBSCAN)
2. Опишите, в каких случаях какой метод предпочтительнее
3. Объясните, как метрики качества помогают выбрать метод и число кластеров
4. Опишите, как кластеризация может использоваться для разметки данных и интерпретации латентных параметров

**Рекомендация:** Используйте результаты ваших экспериментов для аргументации.

**Ваше эссе:**